In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab Notebooks/procesado #CAMBIAR AQUI EL DIRECTORIO

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/procesado


In [ ]:
from itertools import chain
import re
from pkg_resources import resource_stream
import collections
import pandas as pd

In [ ]:
df = pd.read_csv("es.csv", index_col=0)

dfS = df[df.index.str.startswith('-')]
dfP = df[~df.index.str.startswith('-')]

In [ ]:
# función que no usamos porque no usamos el csv
def rima(palabra):
  fonema_rima=dfP.loc[palabra]['pronunciation'].split("ˈ")[1]
  expr="ˈ"+fonema_rima+'$'
  print(expr)
  resultados = dfP[dfP['pronunciation'].str.contains(expr, na=False)].index.tolist()
  return resultados

Función que he añadido para sacar la rima de las palabra (reemplazar la librería de pronouncing)

In [ ]:
def df_rimas(palabra):
  vocales='aeiou'
  tilde=vocales+'ns'
  dict={'a':1, 'e':1, 'i':0, 'o':1, 'u':0}
  numSil=0

  #lleva tilde
  posicion_tilde = re.search("[áéíóú]", palabra)
  if posicion_tilde:
    indice_tilde = posicion_tilde.start()
    final = palabra[indice_tilde:]

  #no lleva tilde
  else:
    final=""

    #saber si es llana o aguda
    if palabra[-1] not in tilde or len(palabra)<=3:
      silabas=1
      #print('aguda')
    else:
      silabas=2
      #print('llana')

    letra = len(palabra)-1
    while numSil<silabas and letra>=0:
      if palabra[letra] in vocales: #nueva sílaba?
        if palabra[letra-1] not in vocales:
          numSil+=1
        elif dict[palabra[letra]]+dict[palabra[letra-1]]>1:
          numSil+=1
      final+=palabra[letra]
      letra-=1
    final=final[::-1]

  return final

Función que busca las rimas de una palabra en el dataframe

In [ ]:
def rimas(palabra):
  fila = df_finales.loc[palabra]
  filas_rimas = df_finales[df_finales["final"] == fila["final"]]
  return filas_rimas.index.tolist()


Creación de df que contendrá todas las palabras de los poemas que queramos usar y la parte de la palabra que contiene la rima.

In [ ]:
df_finales = pd.DataFrame(columns=["palabra", "final"])
df_finales.set_index("palabra", inplace=True)

In [ ]:
versos=[]
with open('rosalia (1).txt') as f: #NOMBRE DE TU TXT CON LAS PALABRAS QUE QUIERAS METER EN EL DATAFRAME
  for verso in f:
      versos.append([v.lower().strip(",.:;¿?¡!") for v in verso.split()])


In [ ]:
versos #mostrar los versos (no es necesario ejecutarlo)

Meter las palabras de la lista de versos en el dataframe:

In [ ]:
for verso in versos:
  for palabra in verso:
    if palabra not in df_finales.index and palabra != '':
    #if palabra not in df_finales[palabra] and palabra!='':
      nueva_fila = pd.DataFrame({'final': [df_rimas(palabra)]}, index=[palabra])
      df_finales = pd.concat([df_finales, nueva_fila])

In [ ]:
print(rimas("flor")) #Ejemplo de palabras que aparecen y riman con flor

['por', 'acusador', 'fulgor', 'amor', 'color', 'ardor', 'calor', 'dolor', 'señor', 'honor', 'flor', 'rumor', '“¿por', 'horror', 'sor', 'engañador', 'desolador', 'mayor', 'asolador', 'rencor', 'candor', 'resplandor', 'verdor', 'redentor', 'estertor', 'rubor']


Dataframe que contiene las palabras y sus partes que riman:

In [ ]:
df_finales

,final
ah,ah
de,e
dolientes,ientes
sauces,auces
rodeada,ada
...,...
mismas,ismas
cadenas,enas
enemigo,igo
libraros,aros


Aqui empieza el código sacado de kaggle: https://www.kaggle.com/code/paultimothymooney/poetry-generator-rnn-markov

In [ ]:
# PARA COMPROBAR EL CONTENIDO (no necesario ejecutarlo)
poeta_file = 'lorca.txt' #CAMBIAR A CÓMO SE LLAMA TU TXT
with open(poeta_file) as f: # The with keyword automatically closes the file when you are done
    print (f.read(1000))

In [ ]:
# PARA COMPROBAR EL CONTENIDO (no necesario ejecutarlo)
poeta_file = 'rosalia (1).txt'
with open(poeta_file) as f: # The with keyword automatically closes the file when you are done
    print (f.read(1000))

In [ ]:
# FUNCIÓN QUE GENERA GRÁFICAS DE LAS PALABRAS MÁS FRECUENTES (no necesario ejecutarlo)
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
def plotWordFrequency(input):
    f = open(poeta_file,'r', encoding='utf-8')
    words = [x for y in [l.split() for l in f.readlines()] for x in y]
    data = sorted([(w, words.count(w)) for w in set(words)], key = lambda x:x[1], reverse=True)[:40]
    most_words = [x[0] for x in data]
    times_used = [int(x[1]) for x in data]
    plt.figure(figsize=(20,10))
    plt.bar(x=sorted(most_words), height=times_used, color = 'grey', edgecolor = 'black',  width=.5)
    plt.xticks(rotation=45, fontsize=18)
    plt.yticks(rotation=0, fontsize=18)
    plt.xlabel('Most Common Words:', fontsize=18)
    plt.ylabel('Number of Occurences:', fontsize=18)
    plt.title('Most Commonly Used Words: %s' % (poeta_file), fontsize=24)
    plt.show()

In [ ]:
#NO NECESARIO EJECUTARLO
poeta_file = 'rosalia (1).txt'
plotWordFrequency(poeta_file)

In [ ]:
#NO NECESARIO EJECUTARLO
poeta_file = 'lorca.txt'
plotWordFrequency(poeta_file)

In [ ]:
#!pip install pronouncing
!pip install markovify
#import pronouncing
import markovify
import re
import random
import numpy as np
import os
import keras
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers.core import Dense

FUNCIONES QUE CREAN Y ENTRENAN EL MODELO:

In [ ]:
def create_network(depth):
	model = Sequential()
	model.add(LSTM(4, input_shape=(2, 2), return_sequences=True))
	for i in range(depth):
		model.add(LSTM(8, return_sequences=True))
	model.add(LSTM(2, return_sequences=True))
	model.summary()
	model.compile(optimizer='rmsprop',
              loss='mse')
	if poeta + ".rap" in os.listdir(".") and train_mode == False:
		model.load_weights(str(poeta + ".rap"))
		print("loading saved network: " + str(poeta) + ".rap")
	return model

In [ ]:
def markov(text_file):
    ######
	read = open(text_file, "r", encoding='utf-8').read()
	text_model = markovify.NewlineText(read)
	return text_model

In [ ]:
def syllables(line):
	count = 0
	for word in line.split(" "):
		vowels = 'aeiouy'
		vowelsT = 'áéíóú'
# 		word = word.lower().strip("!@#$%^&*()_+-={}[];:,.<>/?")
		word = word.lower().strip(",.:;¿?¡!")
		if len(word) > 0:  # Check if word is not empty
			if word[0] in vowels:
				count +=1
			for index in range(1,len(word)):
				if word[index] in vowels and word[index-1] not in vowels: #si vocal no lleva tilde puede formar parte de la misma sílaba si anterior no lleva tilde
					count +=1
				elif word[index] in vowelsT:
					count+=1
			#if word.endswith('e'):
			#	count -= 1
			#if word.endswith('le'): 	HE QUITADO TODO ESTO PORQUE NO SUCEDE EN ESPAÑOL
			#	count+=1
			if count == 0:
				count +=1
	return count / maxsyllables

In [ ]:
def rhymeindex(lyrics):
	if str(poeta) + ".rhymes" in os.listdir(".") and train_mode == False:
		print ("loading saved rhymes from " + str(poeta) + ".rhymes")
		return open(str(poeta) + ".rhymes", "r",encoding='utf-8').read().split("\n")
	else:
		rhyme_master_list = []
		print ("Building list of rhymes:")
		for i in lyrics:
			word = re.sub(r"\W+", '', i.split(" ")[-1]).lower()
			rhymeslist = rimas(word) #FUNCIÓN CREADA PARA LO DE LAS RIMAS EN VEZ DE LA FUNCIÓN DE PRONOUNCING
			rhymeslistends = []
			for i in rhymeslist:
				rhymeslistends.append(i[-2:])
			try:
				rhymescheme = max(set(rhymeslistends), key=rhymeslistends.count)
			except Exception:
				rhymescheme = word[-2:]
			rhyme_master_list.append(rhymescheme)
		rhyme_master_list = list(set(rhyme_master_list))
		reverselist = [x[::-1] for x in rhyme_master_list]
		reverselist = sorted(reverselist)
		rhymelist = [x[::-1] for x in reverselist]
		print("List of Sorted 2-Letter Rhyme Ends:")
		print(rhymelist)
		f = open(str(poeta) + ".rhymes", "w", encoding='utf-8')
		f.write("\n".join(rhymelist))
		f.close()
		return rhymelist

In [ ]:
def rhyme(line, rhyme_list):
	word = re.sub(r"\W+", '', line.split(" ")[-1]).lower()
	rhymeslist = rimas(word) #FUNCIÓN CREADA PAR LO DE LAS RIMAS EN VEZ DE LA FUNCIÓN DE PRONOUNCING
	rhymeslistends = []
	for i in rhymeslist:
		rhymeslistends.append(i[-2:])
	try:
		rhymescheme = max(set(rhymeslistends), key=rhymeslistends.count)
	except Exception:
		rhymescheme = word[-2:]
	try:
		float_rhyme = rhyme_list.index(rhymescheme)
		float_rhyme = float_rhyme / float(len(rhyme_list))
		return float_rhyme
	except Exception:
		float_rhyme = None
		return float_rhyme

In [ ]:
def split_lyrics_file(text_file):
	text = open(text_file, encoding='utf-8').read()
	text = text.split("\n")
	while "" in text:
		text.remove("")
	return text

In [ ]:
def generate_lyrics(text_model, text_file):
	bars = []
	last_words = []
	lyriclength = len(open(text_file,encoding='utf-8').read().split("\n"))
	count = 0
	markov_model = markov(text_file)

	while len(bars) < lyriclength / 9 and count < lyriclength * 2:
		bar = markov_model.make_sentence(max_overlap_ratio = .6, tries=25)#en max_overlap ratio antes ponia 0.49 tries era antes 100
		if type(bar) != type(None) and syllables(bar) < 1:
			def get_last_word(bar):
				last_word = bar.split(" ")[-1]
				if last_word[-1] in "!.?,":
					last_word = last_word[:-1]
				return last_word
			last_word = get_last_word(bar)
			if bar not in bars and last_words.count(last_word) < 3:
				bars.append(bar)
				last_words.append(last_word)
				count += 1
	return bars

In [ ]:
def build_dataset(lines, rhyme_list):
	dataset = []
	line_list = []
	for line in lines:
		line_list = [line, syllables(line), rhyme(line, rhyme_list)]
		dataset.append(line_list)
	x_data = []
	y_data = []
	for i in range(len(dataset) - 3):
		line1 = dataset[i    ][1:]
		line2 = dataset[i + 1][1:]
		line3 = dataset[i + 2][1:]
		line4 = dataset[i + 3][1:]
		x = [line1[0], line1[1], line2[0], line2[1]]
		x = np.array(x)
		x = x.reshape(2,2)
		x_data.append(x)
		y = [line3[0], line3[1], line4[0], line4[1]]
		y = np.array(y)
		y = y.reshape(2,2)
		y_data.append(y)
	x_data = np.array(x_data)
	y_data = np.array(y_data)
	return x_data, y_data

In [ ]:
def compose_rap(lines, rhyme_list, lyrics_file, model):
	rap_vectors = []
	human_lyrics = split_lyrics_file(lyrics_file)
	initial_index = random.choice(range(len(human_lyrics) - 1))
	initial_lines = human_lyrics[initial_index:initial_index + 2]
	starting_input = []
	for line in initial_lines:
		starting_input.append([syllables(line), rhyme(line, rhyme_list)])
	starting_vectors = model.predict(np.array([starting_input]).flatten().reshape(1, 2, 2))
	rap_vectors.append(starting_vectors)
	for i in range(100):
		rap_vectors.append(model.predict(np.array([rap_vectors[-1]]).flatten().reshape(1, 2, 2)))
	return rap_vectors

In [ ]:
def vectors_into_song(vectors, generated_lyrics, rhyme_list):
	print ("\n\n")
	print ("Writing verse:")
	print ("\n\n")
	def last_word_compare(rap, line2):
		penalty = 0
		for line1 in rap:
			word1 = line1.split(" ")[-1]
			word2 = line2.split(" ")[-1]
			while word1[-1] in "?!,. ":
				word1 = word1[:-1]
			while word2[-1] in "?!,. ":
				word2 = word2[:-1]
			if word1 == word2:
				penalty += 0.2
		return penalty
	def calculate_score(vector_half, syllables, rhyme, penalty):
		desired_syllables = vector_half[0]
		desired_rhyme = vector_half[1]
		desired_syllables = desired_syllables * maxsyllables
		desired_rhyme = desired_rhyme * len(rhyme_list)
		score = 1.0 - abs(float(desired_syllables) - float(syllables)) + abs(float(desired_rhyme) - float(rhyme)) - penalty
		return score
	dataset = []
	for line in generated_lyrics:
		line_list = [line, syllables(line), rhyme(line, rhyme_list)]
		dataset.append(line_list)
	rap = []
	vector_halves = []
	for vector in vectors:
		vector_halves.append(list(vector[0][0]))
		vector_halves.append(list(vector[0][1]))
	for vector in vector_halves:
		scorelist = []
		for item in dataset:
			line = item[0]
			if len(rap) != 0:
				penalty = last_word_compare(rap, line)
			else:
				penalty = 0
			total_score = calculate_score(vector, item[1], item[2], penalty)
			score_entry = [line, total_score]
			scorelist.append(score_entry)
		fixed_score_list = [0]
		for score in scorelist:
			fixed_score_list.append(float(score[1]))
		max_score = max(fixed_score_list)
		for item in scorelist:
			if item[1] == max_score:
				rap.append(item[0])
				print (str(item[0]))
				for i in dataset:
					if item[0] == i[0]:
						dataset.remove(i)
						break
				break
	return rap

ENTRENAMIENTO:

In [ ]:
def train(x_data, y_data, model):
	model.fit(np.array(x_data), np.array(y_data),
			  batch_size=2,
			  epochs=5,
			  verbose=1)
	model.save_weights(poeta + ".rap")

In [ ]:
def main(depth, train_mode):
	model = create_network(depth)
	text_model = markov(text_file)
	if train_mode == True:
		bars = split_lyrics_file(text_file)
	if train_mode == False:
		bars = generate_lyrics(text_model, text_file)
	rhyme_list = rhymeindex(bars)
	if train_mode == True:
		x_data, y_data = build_dataset(bars, rhyme_list)
		train(x_data, y_data, model)
	if train_mode == False:
		vectors = compose_rap(bars, rhyme_list, text_file, model)
		rap = vectors_into_song(vectors, bars, rhyme_list)
		f = open(rap_file, "w", encoding='utf-8')
		for bar in rap:
			f.write(bar)
			f.write("\n")

In [ ]:
depth = 4
maxsyllables = 12
poeta = "poeta"
rap_file = "poema1.txt"

ESTO ES LO QUE TARDA UN MONTONAZO:

In [ ]:
maxsyllables = 12
text_file = "rosalia1.txt"
train_mode = True
main(depth, train_mode)
train_mode = False
main(depth, train_mode)

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_12 (LSTM)              (None, 2, 4)              112       
                                                                 
 lstm_13 (LSTM)              (None, 2, 8)              416       
                                                                 
 lstm_14 (LSTM)              (None, 2, 8)              544       
                                                                 
 lstm_15 (LSTM)              (None, 2, 8)              544       
                                                                 
 lstm_16 (LSTM)              (None, 2, 8)              544       
                                                                 
 lstm_17 (LSTM)              (None, 2, 2)              88        
                                                                 
Total params: 2,248
Trainable params: 2,248
Non-traina

KeyboardInterrupt: ignored

In [ ]:
text_file = "lorca.txt"
maxsyllables = 8
train_mode = True
main(depth, train_mode)
train_mode = False
main(depth, train_mode)

In [ ]:
filenames = ['rosalia.txt', 'lorca.txt']
with open('combined.txt', 'w') as outfile:
    for fname in filenames:
        with open(fname) as infile:
            for line in infile:
                outfile.write(line)

In [ ]:
maxsyllables = 8
text_file = "combined.txt"
train_mode = True
main(depth, train_mode)
train_mode = False
main(depth, train_mode)